# MapReduce Algorithms

## Learning Objectives

By the end of this notebook you will be able to:

1. **Express** relational algebra operations (selection, projection, join, grouping) as MapReduce jobs
2. **Implement** matrix-vector multiplication in the MapReduce model
3. **Recognise** the pattern: what goes in the key determines how shuffle groups data

## Relational Operations in MapReduce

A relation $R$ is a table: a set of tuples $(a_1, a_2, \ldots, a_k)$.
MapReduce can implement every relational algebra operator.

| Operator | SQL equivalent | MapReduce strategy |
|----------|---------------|-------------------|
| Selection $\sigma_C(R)$ | `WHERE C` | Map filters rows; Reduce is identity |
| Projection $\pi_S(R)$ | `SELECT cols` | Map drops columns; Reduce deduplicates |
| Union $R \cup S$ | `UNION` | Map tags tuples with relation name; Reduce outputs distinct tuples |
| Natural Join $R \bowtie S$ | `JOIN ON shared cols` | Map keys on join attribute; Reduce pairs matching tuples |
| Grouping & Aggregation | `GROUP BY + agg` | Map keys on group-by columns; Reduce applies aggregation |

## 1. Selection

$$\sigma_C(R) = \{t \in R \mid C(t) \text{ is true}\}$$

**Map:** for each tuple $t$, if $C(t)$ then emit $(t, t)$  
**Reduce:** identity — emit each value as-is

Since every tuple is independent, selection is pure Map with no Reduce needed.

In [ ]:
def selection(relation, condition):
    """Filter rows satisfying condition. No shuffle needed — pure Map."""
    return [row for row in relation if condition(row)]


employees = [
    {"name": "Alice", "dept": "Eng",   "salary": 120_000},
    {"name": "Bob",   "dept": "Sales", "salary":  80_000},
    {"name": "Carol", "dept": "Eng",   "salary": 110_000},
    {"name": "Dave",  "dept": "Sales", "salary":  95_000},
]

result = selection(employees, lambda r: r["dept"] == "Eng")
for row in result:
    print(row)

## 2. Projection

$$\pi_S(R) = \{(t.a \mid a \in S) \mid t \in R\}$$

**Map:** for each tuple $t$, emit $(\pi_S(t), \pi_S(t))$ — key and value are the projected tuple  
**Reduce:** for each key (projected tuple) emit it once — this deduplicates

Without duplicates (when $S$ includes a key), Reduce is simply identity.

In [ ]:
from collections import defaultdict


def projection(relation, columns):
    """Project onto columns and deduplicate."""
    # Map phase
    intermediate = {}
    for row in relation:
        projected = tuple(row[c] for c in columns)
        intermediate[projected] = True   # dedup via key

    # Reduce phase (identity after dedup)
    return [dict(zip(columns, t)) for t in intermediate]


result = projection(employees, ["dept"])
for row in result:
    print(row)

## 3. Natural Join

$$R(A, B) \bowtie S(B, C) = \{(a, b, c) \mid (a, b) \in R \text{ and } (b, c) \in S\}$$

**Map:**
- For each tuple $(a, b) \in R$: emit key $= b$, value $= (R, a)$
- For each tuple $(b, c) \in S$: emit key $= b$, value $= (S, c)$

**Shuffle:** groups all values with the same join attribute $b$

**Reduce:** for each $b$, take the cross-product of $R$-values and $S$-values

In [ ]:
def natural_join(R, R_join_col, R_other_cols,
                 S, S_join_col, S_other_cols):
    """Natural join of two relations on their shared column."""
    # Map phase
    grouped = defaultdict(lambda: {"R": [], "S": []})

    for row in R:
        key = row[R_join_col]
        val = tuple(row[c] for c in R_other_cols)
        grouped[key]["R"].append(val)

    for row in S:
        key = row[S_join_col]
        val = tuple(row[c] for c in S_other_cols)
        grouped[key]["S"].append(val)

    # Reduce phase — cross-product within each group
    results = []
    all_cols = R_other_cols + [R_join_col] + S_other_cols
    for join_val, sides in grouped.items():
        for r_tuple in sides["R"]:
            for s_tuple in sides["S"]:
                row = dict(zip(all_cols, r_tuple + (join_val,) + s_tuple))
                results.append(row)
    return results


orders = [
    {"order_id": 1, "customer_id": 10, "amount": 250},
    {"order_id": 2, "customer_id": 20, "amount": 130},
    {"order_id": 3, "customer_id": 10, "amount": 480},
]
customers = [
    {"customer_id": 10, "name": "Alice"},
    {"customer_id": 20, "name": "Bob"},
]

result = natural_join(
    orders,    "customer_id", ["order_id", "amount"],
    customers, "customer_id", ["name"],
)
for row in result:
    print(row)

## 4. Grouping & Aggregation

$$\gamma_{G, \theta(B)}(R)$$

Groups tuples by columns $G$, then applies aggregation $\theta$ (sum, count, max, …) over column $B$.

**Map:** for each tuple $t$, emit key $= t[G]$, value $= t[B]$  
**Reduce:** for each key, apply $\theta$ to the list of values

In [ ]:
def group_by_agg(relation, group_cols, agg_col, agg_fn):
    # Map
    grouped = defaultdict(list)
    for row in relation:
        key = tuple(row[c] for c in group_cols)
        grouped[key].append(row[agg_col])

    # Reduce
    return [
        dict(list(zip(group_cols, k)) + [("result", agg_fn(vs))])
        for k, vs in sorted(grouped.items())
    ]


result = group_by_agg(employees, ["dept"], "salary", sum)
for row in result:
    print(row)

## 5. Matrix-Vector Multiplication

For $\mathbf{y} = M \mathbf{x}$ where $M$ is $m \times n$:

$$y_i = \sum_{j=0}^{n-1} M_{ij} \cdot x_j$$

**Map:** for each entry $M_{ij} = v$, emit key $= i$, value $= v \cdot x_j$  
**Reduce:** for each row $i$, sum all partial products $\to y_i$

This requires $M$ and $x$ to both be available to the Map tasks.
For very large $x$, it is broadcast; for very large $M$, it is stored in the distributed filesystem.

In [ ]:
import numpy as np
from collections import defaultdict


def matvec_mapreduce(M_sparse, x):
    """
    Sparse matrix-vector multiplication y = M @ x via MapReduce.
    M_sparse : list of (i, j, value) triples
    x        : dense vector (numpy array or list)
    """
    # Map: (i, j, v) -> (i, v * x[j])
    intermediate = []
    for i, j, v in M_sparse:
        intermediate.append((i, v * x[j]))

    # Shuffle
    grouped = defaultdict(list)
    for i, partial in intermediate:
        grouped[i].append(partial)

    # Reduce: sum partial products per row
    n_rows = max(grouped) + 1
    return np.array([sum(grouped[i]) for i in range(n_rows)])


M = np.array([[2, 0, 1],
              [0, 3, 4],
              [5, 0, 0]], dtype=float)
x = np.array([1.0, 2.0, 3.0])

M_sparse = [(i, j, M[i,j])
            for i in range(M.shape[0])
            for j in range(M.shape[1])
            if M[i,j] != 0]

y_mr = matvec_mapreduce(M_sparse, x)
y_np = M @ x
print("MapReduce:", y_mr)
print("NumPy:    ", y_np)
print("Match:", np.allclose(y_mr, y_np))

## Summary

| Algorithm | Map key | Reduce operation |
|-----------|---------|-----------------|
| Selection | tuple (or discard) | identity / none |
| Projection | projected tuple | deduplicate |
| Natural Join | join attribute value | cross-product of matching tuples |
| Group-by Agg | group-by columns | apply aggregation function |
| Matrix-vector multiply | row index $i$ | sum partial products |

The **key** determines how data is grouped in the shuffle phase.
Choosing the right key is the core design decision in every MapReduce algorithm.